# 02 — Data Cleaning & Structuring
### UPI Transaction Trends India 2024

**Objective:** Take the raw synthetic transaction dataset and produce a clean,
analysis-ready file. Every cleaning decision is documented with a business reason.

**Issues to fix:**
| # | Issue | Count | Fix |
|---|---|---|---|
| 1 | Duplicate rows | ~1,000 | Drop exact duplicates, keep first |
| 2 | Inconsistent casing in `upi_app` | ~500 | Standardise to Title Case |
| 3 | Missing `city` values | ~510 | Impute as 'Unknown' |
| 4 | Missing `merchant_category` for P2P rows | ~23,000 | Expected — fill with 'N/A (P2P)' |
| 5 | Amount outliers (> ₹1,00,000) | ~153 | Flag & cap for analysis |

**Input:**  `data/raw/upi_transactions_2024.csv`  
**Output:** `data/processed/transactions_clean.csv`  
           `data/processed/monthly_summary.csv`


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("Libraries loaded ✓")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy  version: {np.__version__}")


## 2. Load Raw Data

In [ ]:
RAW_PATH       = '../data/raw/upi_transactions_2024.csv'
CLEAN_PATH     = '../data/processed/transactions_clean.csv'
SUMMARY_PATH   = '../data/processed/monthly_summary.csv'

os.makedirs('../data/processed', exist_ok=True)

df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()   # Always work on a copy — never touch the raw file

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head(3)


## 3. Data Audit (Before Cleaning)
Run a full audit before touching anything. This becomes your "before" snapshot
for the portfolio README and interview talking points.


In [ ]:
print("=" * 55)
print("DATA AUDIT — RAW DATASET")
print("=" * 55)

print(f"\nShape : {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n── Column Data Types ──────────────────────────────────")
print(df.dtypes)

print("\n── Null Values ────────────────────────────────────────")
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
audit_nulls = pd.DataFrame({'null_count': nulls, 'null_%': nulls_pct})
print(audit_nulls[audit_nulls['null_count'] > 0])

print("\n── Duplicates ─────────────────────────────────────────")
print(f"  Duplicate transaction_ids : {df['transaction_id'].duplicated().sum():,}")
print(f"  Fully duplicate rows      : {df.duplicated().sum():,}")

print("\n── upi_app unique values ──────────────────────────────")
print(df['upi_app'].value_counts())

print("\n── Amount statistics ──────────────────────────────────")
print(df['amount_inr'].describe().apply(lambda x: f'{x:,.2f}'))

print("\n── Amount outliers > ₹1,00,000 ────────────────────────")
outliers = df[df['amount_inr'] > 100_000]
print(f"  Count : {len(outliers):,}")
print(f"  Max   : ₹{df['amount_inr'].max():,.2f}")


In [ ]:
# Quick visual: Amount distribution (raw)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: full distribution (shows the skew)
axes[0].hist(df['amount_inr'], bins=100, color='steelblue', edgecolor='none', alpha=0.7)
axes[0].set_title('Amount Distribution — Raw (all values)', fontsize=12)
axes[0].set_xlabel('Amount (₹)')
axes[0].set_ylabel('Frequency')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: capped at ₹10,000 to see the real shape
axes[1].hist(df[df['amount_inr'] <= 10_000]['amount_inr'], bins=100,
             color='steelblue', edgecolor='none', alpha=0.7)
axes[1].set_title('Amount Distribution — Capped at ₹10,000', fontsize=12)
axes[1].set_xlabel('Amount (₹)')
axes[1].set_ylabel('Frequency')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Raw Data: Amount Distribution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/audit_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved ✓")


## 4. Cleaning Steps

Each step follows the same pattern:
1. Identify the issue with a count
2. Apply the fix
3. Verify with an after-count

This is intentional — it shows reviewers you validated every step.


### 4.1 Fix #1 — Remove Duplicate Rows

In [ ]:
# ── Before ──────────────────────────────────────────────
before = len(df)
dup_count = df.duplicated().sum()
print(f"Before : {before:,} rows | Duplicates found : {dup_count:,}")

# ── Fix: drop full duplicates, keep the first occurrence ─
# Business reason: a transaction is uniquely identified by
# transaction_id + timestamp. Keeping the first occurrence
# is the standard approach when no other dedup signal exists.
df = df.drop_duplicates().reset_index(drop=True)

# ── After ────────────────────────────────────────────────
after = len(df)
print(f"After  : {after:,} rows | Removed : {before - after:,} rows")
assert df.duplicated().sum() == 0, "Duplicates still exist!"
print("✓ No duplicates remaining")


### 4.2 Fix #2 — Standardise `upi_app` Casing

In [ ]:
# ── Before ──────────────────────────────────────────────
print("Before — unique upi_app values:")
print(df['upi_app'].value_counts().to_string())

# ── Fix: strip whitespace + title case ───────────────────
# Business reason: 'PHONEPE' and 'PhonePe' are the same app.
# Inconsistent casing breaks GROUP BY in SQL and value_counts()
# in Python, producing misleading market share numbers.
df['upi_app'] = df['upi_app'].str.strip().str.title()

# ── After ────────────────────────────────────────────────
print("\nAfter — unique upi_app values:")
print(df['upi_app'].value_counts().to_string())
print(f"\nUnique app count: {df['upi_app'].nunique()} (expected 6)")
assert df['upi_app'].nunique() == 6, "Unexpected app count after standardisation!"
print("✓ upi_app standardised")


### 4.3 Fix #3 — Handle Missing `city` Values

In [ ]:
# ── Before ──────────────────────────────────────────────
missing_city = df['city'].isnull().sum()
print(f"Before : {missing_city:,} missing city values ({missing_city/len(df)*100:.2f}%)")

# ── Fix: impute with 'Unknown' ────────────────────────────
# Business reason: We cannot infer the city from other columns.
# Dropping these rows would lose valid transaction data.
# Labelling as 'Unknown' is transparent and keeps the rows usable
# for amount/volume analysis while clearly marking the gap.
df['city'] = df['city'].fillna('Unknown')

# ── After ────────────────────────────────────────────────
print(f"After  : {df['city'].isnull().sum()} missing city values")
print(f"'Unknown' city count : {(df['city'] == 'Unknown').sum():,}")
print("✓ Missing city values handled")


### 4.4 Fix #4 — Handle Missing `merchant_category`

In [ ]:
# ── Before ──────────────────────────────────────────────
missing_mc = df['merchant_category'].isnull().sum()
print(f"Before : {missing_mc:,} missing merchant_category values")

# Check breakdown: P2P vs P2M
p2p_missing = df[(df['transaction_type'] == 'P2P') & (df['merchant_category'].isnull())].shape[0]
p2m_missing = df[(df['transaction_type'] == 'P2M') & (df['merchant_category'].isnull())].shape[0]
print(f"  Of which P2P rows : {p2p_missing:,}  (expected — P2P has no merchant)")
print(f"  Of which P2M rows : {p2m_missing:,}  (these need attention)")

# ── Fix ───────────────────────────────────────────────────
# For P2P transactions: 'N/A (P2P)' — correct by design
df.loc[(df['transaction_type'] == 'P2P') & (df['merchant_category'].isnull()),
       'merchant_category'] = 'N/A (P2P)'

# For P2M transactions with missing category: 'Uncategorised'
df.loc[(df['transaction_type'] == 'P2M') & (df['merchant_category'].isnull()),
       'merchant_category'] = 'Uncategorised'

# ── After ────────────────────────────────────────────────
print(f"\nAfter  : {df['merchant_category'].isnull().sum()} missing merchant_category values")
print(f"Unique categories now: {df['merchant_category'].nunique()}")
print("✓ merchant_category cleaned")


### 4.5 Fix #5 — Flag & Cap Amount Outliers

In [ ]:
# ── Before ──────────────────────────────────────────────
outlier_threshold = 100_000  # ₹1,00,000
outliers = df[df['amount_inr'] > outlier_threshold]
print(f"Outliers above ₹{outlier_threshold:,} : {len(outliers):,} rows")
print(f"Max value : ₹{df['amount_inr'].max():,.2f}")

# ── Fix: flag (don't delete) ──────────────────────────────
# Business reason: Transactions above ₹1 lakh could be legitimate
# high-value transfers (rent, business payments).
# Best practice is to FLAG them for separate analysis
# rather than silently deleting or capping them.
# We add an 'is_outlier' boolean column and exclude from standard analysis.
df['is_outlier'] = df['amount_inr'] > outlier_threshold

# ── After ────────────────────────────────────────────────
print(f"\nOutlier flag added. Flagged rows : {df['is_outlier'].sum():,}")
print(f"Clean rows (is_outlier=False)     : {(~df['is_outlier']).sum():,}")
print("✓ Outliers flagged")


## 5. Data Type Corrections

In [ ]:
# ── Convert timestamp and date to proper datetime types ──
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date']      = pd.to_datetime(df['date'])

# ── Ensure categoricals use category dtype (memory efficient) ─
cat_cols = ['transaction_type', 'upi_app', 'city', 'status',
            'merchant_category', 'day_of_week', 'month_name']
for col in cat_cols:
    df[col] = df[col].astype('category')

# ── Verify ────────────────────────────────────────────────
print("Updated dtypes:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


## 6. Feature Engineering
Add useful derived columns that will power the SQL queries and dashboard.


In [ ]:
# Quarter
df['quarter'] = df['timestamp'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})

# Day type
df['day_type'] = df['day_of_week'].astype(str).apply(
    lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday'
)

# Time of day bucket
def time_bucket(hour):
    if   0  <= hour < 6:  return 'Night (12am–6am)'
    elif 6  <= hour < 12: return 'Morning (6am–12pm)'
    elif 12 <= hour < 18: return 'Afternoon (12pm–6pm)'
    else:                  return 'Evening (6pm–12am)'

df['time_of_day'] = df['hour'].apply(time_bucket).astype('category')

# Amount bracket (for grouping in dashboard)
def amount_bracket(amt):
    if   amt <= 100:    return '₹0–100'
    elif amt <= 500:    return '₹101–500'
    elif amt <= 2000:   return '₹501–2,000'
    elif amt <= 10000:  return '₹2,001–10,000'
    elif amt <= 100000: return '₹10,001–1,00,000'
    else:               return '> ₹1,00,000'

df['amount_bracket'] = df['amount_inr'].apply(amount_bracket).astype('category')

print("New columns added:", ['quarter', 'day_type', 'time_of_day', 'amount_bracket'])
print()
print("Quarter distribution:")
print(df['quarter'].value_counts().sort_index())
print()
print("Time of day distribution:")
print(df['time_of_day'].value_counts())
print()
print("Amount bracket distribution:")
print(df['amount_bracket'].value_counts())


## 7. Post-Clean Audit

In [ ]:
print("=" * 55)
print("DATA AUDIT — CLEANED DATASET")
print("=" * 55)

print(f"\nShape : {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n── Null values remaining ──────────────────────────────")
nulls_after = df.isnull().sum()
print(nulls_after[nulls_after > 0] if nulls_after.sum() > 0 else "  None ✓")

print("\n── Duplicate rows remaining ───────────────────────────")
print(f"  {df.duplicated().sum()} ✓")

print("\n── upi_app values ─────────────────────────────────────")
print(df['upi_app'].value_counts())

print("\n── Amount stats (clean rows only) ─────────────────────")
clean_amounts = df[~df['is_outlier']]['amount_inr']
print(clean_amounts.describe().apply(lambda x: f'₹{x:,.2f}'))

print("\n── New feature columns ────────────────────────────────")
for col in ['quarter', 'day_type', 'time_of_day', 'amount_bracket']:
    print(f"  {col} — {df[col].nunique()} unique values")


## 8. Build Monthly Summary Table
Aggregate the cleaned transaction data into a monthly summary.
This will be joined with the real NPCI data in the EDA notebook.


In [ ]:
# Filter out outliers for the summary
df_clean = df[~df['is_outlier']].copy()

monthly_summary = df_clean.groupby(['month', 'month_name'], observed=True).agg(
    total_transactions   = ('transaction_id', 'count'),
    total_value_inr      = ('amount_inr', 'sum'),
    avg_transaction_inr  = ('amount_inr', 'mean'),
    median_txn_inr       = ('amount_inr', 'median'),
    p2p_transactions     = ('transaction_type', lambda x: (x == 'P2P').sum()),
    p2m_transactions     = ('transaction_type', lambda x: (x == 'P2M').sum()),
    success_count        = ('status', lambda x: (x.astype(str) == 'Success').sum()),
    failed_count         = ('status', lambda x: (x.astype(str) == 'Failed').sum()),
    unique_cities        = ('city', 'nunique'),
).reset_index().sort_values('month')

# Derived metrics
monthly_summary['success_rate_%']    = (monthly_summary['success_count'] /
                                         monthly_summary['total_transactions'] * 100).round(2)
monthly_summary['p2p_share_%']       = (monthly_summary['p2p_transactions'] /
                                         monthly_summary['total_transactions'] * 100).round(2)
monthly_summary['p2m_share_%']       = (monthly_summary['p2m_transactions'] /
                                         monthly_summary['total_transactions'] * 100).round(2)
monthly_summary['mom_txn_growth_%']  = monthly_summary['total_transactions'].pct_change().mul(100).round(2)

# Format large numbers for readability
monthly_summary['total_value_cr']    = (monthly_summary['total_value_inr'] / 1e7).round(2)

print("Monthly Summary (synthetic data):")
print(monthly_summary[['month_name', 'total_transactions', 'total_value_cr',
                         'avg_transaction_inr', 'success_rate_%',
                         'p2m_share_%', 'mom_txn_growth_%']].to_string(index=False))


## 9. Save Cleaned Files

In [ ]:
# Save main cleaned dataset
df.to_csv(CLEAN_PATH, index=False)
print(f"✓ Cleaned transactions saved : {CLEAN_PATH}")
print(f"  Rows: {len(df):,}  |  Columns: {df.shape[1]}")

# Save monthly summary
monthly_summary.to_csv(SUMMARY_PATH, index=False)
print(f"\n✓ Monthly summary saved     : {SUMMARY_PATH}")
print(f"  Rows: {len(monthly_summary)}  |  Columns: {monthly_summary.shape[1]}")


## 10. Cleaning Summary — Interview-Ready

In [ ]:
summary = {
    "Fix": [
        "Remove duplicates",
        "Standardise upi_app casing",
        "Impute missing city",
        "Fix merchant_category nulls",
        "Flag amount outliers"
    ],
    "Rows Affected": [
        f"~{df_raw.duplicated().sum():,}",
        "~500",
        f"~{df_raw['city'].isnull().sum():,}",
        f"~{df_raw['merchant_category'].isnull().sum():,}",
        f"{df['is_outlier'].sum():,}"
    ],
    "Method": [
        "drop_duplicates(keep='first')",
        "str.strip().str.title()",
        "fillna('Unknown')",
        "fillna by txn_type logic",
        "Boolean flag column added"
    ],
    "Business Reason": [
        "Duplicate TXN IDs inflate volume & value KPIs",
        "Inconsistent casing breaks GROUP BY aggregations",
        "Dropping rows loses valid transaction signals",
        "P2P has no merchant; P2M nulls marked Uncategorised",
        "High-value transfers are valid; transparent exclusion"
    ]
}

summary_df = pd.DataFrame(summary)
print("CLEANING DECISIONS LOG")
print("=" * 100)
print(summary_df.to_string(index=False))
print()
print(f"Final dataset: {len(df):,} rows × {df.shape[1]} columns")
print(f"Raw dataset  : {len(df_raw):,} rows × {df_raw.shape[1]} columns")
print(f"Rows removed : {len(df_raw) - len(df):,} (duplicates only)")
print(f"Columns added: {df.shape[1] - df_raw.shape[1]} (quarter, day_type, time_of_day, amount_bracket, is_outlier)")


---
**Next:** `03_eda_visualisation.ipynb` — Exploratory Data Analysis with charts
